# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/irene1407/ML-1/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

### Signal 1: CTR vs. Average Position

I will investigate whether content items with worse average search positions tend to have lower click-through rates.

This signal is linked to FlyRank's CTR-fix logic. If the relationship is not consistent, I will treat it cautiously rather than assume that position explains CTR.

### Signal 2: Search Impressions

I will investigate how impressions are distributed across content items and whether high-impression items represent meaningful visibility opportunities.

This signal is linked to FlyRank's quick-win logic. High impressions alone do not guarantee that an item needs improvement.

### Provisional Rule Idea

If both signals are supported by the data, I plan to prioritize content items with substantial impressions and CTR below the typical CTR for their average-position group.

The rule will use March 2026 data only. The signal checks will determine whether this idea is worth encoding.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
%pip install -q duckdb

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip available: 22.3.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import duckdb
import pandas as pd
import numpy as np
from pathlib import Path
from IPython.display import display
import getpass

con = duckdb.connect()

hf_token = getpass.getpass("Enter your Hugging Face READ token: ")

con.execute("""
    CREATE OR REPLACE SECRET hf (
        TYPE huggingface,
        TOKEN ?
    )
""", [hf_token])

print("DuckDB connection ready!")

DuckDB connection ready!


In [3]:
MARCH = """
read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)
"""

march_query = f"""
SELECT
    client_hash_id,
    content_hash_id,
    SUM(gsc_impressions) AS impressions,
    SUM(gsc_clicks) AS clicks,
    SUM(gsc_avg_position * gsc_impressions)
        / NULLIF(SUM(gsc_impressions), 0) AS avg_position,
    SUM(gsc_clicks) * 100.0
        / NULLIF(SUM(gsc_impressions), 0) AS march_ctr,
    COUNT(DISTINCT report_date) AS active_days
FROM {MARCH}
WHERE gsc_data_available IS TRUE
  AND gsc_impressions > 0
  AND gsc_avg_position > 0
GROUP BY client_hash_id, content_hash_id
"""

df = con.execute(march_query).df()

print("March feature frame shape:", df.shape)
print("Missing values:", df.isna().sum().sum())
display(df.head())

March feature frame shape: (175304, 7)
Missing values: 0


,client_hash_id,content_hash_id,impressions,clicks,avg_position,march_ctr,active_days
0,client_e547b89c05043229,content_a3963755bddf1a48,2783.0,3.0,8.637442,0.107797,29
1,client_e547b89c05043229,content_76f4c2e527289c98,1036.0,4.0,9.896718,0.386100,29
2,client_e547b89c05043229,content_86a7f6ca601efe37,498.0,0.0,14.172691,0.000000,29
3,client_e547b89c05043229,content_e477dc8c17c791c3,2397.0,10.0,5.603671,0.417188,29
4,client_e547b89c05043229,content_241ab01426122c1b,2411.0,5.0,20.617586,0.207383,29


In [4]:
# Position groups
position_bins = [0, 3, 10, 20, 50, np.inf]
position_labels = ["1-3", "4-10", "11-20", "21-50", "50+"]

df["position_bucket"] = pd.cut(
    df["avg_position"],
    bins=position_bins,
    labels=position_labels,
    right=True
)

# Signal 1: CTR by average-position group
signal_1 = (
    df.groupby("position_bucket", observed=False)
      .agg(
          n=("content_hash_id", "count"),
          median_ctr=("march_ctr", "median"),
          mean_ctr=("march_ctr", "mean")
      )
      .reset_index()
)

print("SIGNAL 1: CTR by average-position group")
display(signal_1.round(3))

# Signal 2: impressions by visibility quartile
# Rank first ensures that qcut can form four groups even with tied values.
df["impression_bucket"] = pd.qcut(
    df["impressions"].rank(method="first"),
    q=4,
    labels=["Q1 - Lowest", "Q2", "Q3", "Q4 - Highest"]
)

signal_2 = (
    df.groupby("impression_bucket", observed=False)
      .agg(
          n=("content_hash_id", "count"),
          median_impressions=("impressions", "median"),
          median_ctr=("march_ctr", "median"),
          mean_ctr=("march_ctr", "mean")
      )
      .reset_index()
)

print("SIGNAL 2: Metrics by impressions group")
display(signal_2.round(3))

SIGNAL 1: CTR by average-position group


,position_bucket,n,median_ctr,mean_ctr
0,1-3,15043,0.069,0.998
1,4-10,83546,0.000,0.474
2,11-20,30200,0.000,0.331
3,21-50,33123,0.000,0.228
4,50+,13392,0.000,0.085


SIGNAL 2: Metrics by impressions group


,impression_bucket,n,median_impressions,median_ctr,mean_ctr
0,Q1 - Lowest,43826,4.0,0.000,0.857
1,Q2,43826,64.0,0.000,0.283
2,Q3,43826,423.0,0.000,0.239
3,Q4 - Highest,43826,3040.0,0.195,0.296


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

### Signal Verdicts

**Signal 1: CTR vs. Average Position**

Verdict: [YOUR ONE-WORD VERDICT]

Evidence: [Describe the pattern in the bucket table, including relevant n values.]

**Signal 2: Search Impressions**

Verdict: [YOUR ONE-WORD VERDICT]

Evidence: [Describe the pattern in the bucket table, including relevant n values.]

### Rule Decision

[Explain whether the evidence supports using these signals in the baseline rule. Mention any caveats.]

## 2. Build the Ranked Queue

The rule uses two conditions:

- High visibility: impressions are at or above the 75th percentile.
- Below-typical CTR: March CTR is below the median CTR for the item's position group.

For qualifying items:

`score = impressions × (position-group median CTR - item CTR)`

A higher score means a larger estimated visibility-weighted CTR gap.

Items that do not meet both conditions receive a score of zero.

The rule assigns one reason code, `high_visibility_low_ctr`, to qualifying items and recommends `review_title_meta`.

The score is a prioritization heuristic, not a prediction of guaranteed improvement.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Calculate typical CTR for each position group
position_baseline = (
    df.groupby("position_bucket", observed=False)["march_ctr"]
      .median()
      .rename("expected_ctr")
      .reset_index()
)

df = df.merge(
    position_baseline,
    on="position_bucket",
    how="left"
)

# High-impression threshold
impression_threshold = df["impressions"].quantile(0.75)

# Calculate the gap below typical CTR
df["ctr_gap"] = (
    df["expected_ctr"] - df["march_ctr"]
).clip(lower=0)

# Apply the rule
df["qualifies"] = (
    (df["impressions"] >= impression_threshold)
    & (df["march_ctr"] < df["expected_ctr"])
)

# Transparent score
df["score"] = np.where(
    df["qualifies"],
    df["impressions"] * df["ctr_gap"],
    0
)

# One reason code and one action label
df["reason_code"] = np.where(
    df["qualifies"],
    "high_visibility_low_ctr",
    None
)

df["action"] = np.where(
    df["qualifies"],
    "review_title_meta",
    "monitor"
)

# Rank the queue
queue = df.sort_values(
    ["score", "impressions"],
    ascending=[False, False]
).copy()

queue.insert(0, "rank", range(1, len(queue) + 1))

print("Impression threshold:", round(impression_threshold, 2))
print("Total content items:", len(queue))
print("Items selected for review:", int(queue["qualifies"].sum()))

display(
    queue[
        [
            "rank",
            "client_hash_id",
            "content_hash_id",
            "impressions",
            "avg_position",
            "march_ctr",
            "expected_ctr",
            "score",
            "reason_code",
            "action"
        ]
    ].head(20).round(3)
)

Impression threshold: 1052.0
Total content items: 175304
Items selected for review: 897


,rank,client_hash_id,content_hash_id,impressions,avg_position,march_ctr,expected_ctr,score,reason_code,action
88684,1,client_23a62021009f63c4,content_44f34c0a90047651,212404.0,0.666,0.011,0.069,12262.034,high_visibility_low_ctr,review_title_meta
17165,2,client_73cda7b4e4f265ea,content_8e1334d6356668e3,134984.0,2.693,0.001,0.069,9217.809,high_visibility_low_ctr,review_title_meta
162352,3,client_73cda7b4e4f265ea,content_fec55986a1868d62,124075.0,0.308,0.001,0.069,8464.772,high_visibility_low_ctr,review_title_meta
49729,4,client_73cda7b4e4f265ea,content_9c057b66c30a3abb,83832.0,0.116,0.001,0.069,5686.838,high_visibility_low_ctr,review_title_meta
53756,5,client_23a62021009f63c4,content_bf078007df823490,44707.0,1.400,0.000,0.069,3086.079,high_visibility_low_ctr,review_title_meta
173139,6,client_1a730cb2640a1abf,content_d61fc394d10cba41,38000.0,2.363,0.003,0.069,2523.102,high_visibility_low_ctr,review_title_meta
99961,7,client_62f4a7e64f5e0096,content_fc67675904376267,60172.0,2.126,0.030,0.069,2353.613,high_visibility_low_ctr,review_title_meta
149379,8,client_e547b89c05043229,content_306bc78dff1eb683,80821.0,1.444,0.043,0.069,2078.992,high_visibility_low_ctr,review_title_meta
41069,9,client_e547b89c05043229,content_dc91779c3d085398,25625.0,2.389,0.004,0.069,1668.868,high_visibility_low_ctr,review_title_meta
62516,10,client_e547b89c05043229,content_757b1fa67827358d,31575.0,2.855,0.019,0.069,1579.590,high_visibility_low_ctr,review_title_meta


In [6]:
output_dir = Path.cwd().resolve().parents[1] / "work" / "outputs"
output_dir.mkdir(parents=True, exist_ok=True)

output_path = output_dir / "baseline_action_score.csv"

queue.to_csv(output_path, index=False)

print("Ranked queue saved to:", output_path)
print("File exists:", output_path.exists())

Ranked queue saved to: C:\Users\LENOVO\Downloads\ML-1\work\outputs\baseline_action_score.csv
File exists: True


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

## 3. Top-20 Review

I will manually inspect the twenty highest-ranked items.

For each item, I will record:

- **Action:** What should be reviewed?
- **Reason:** Why did the item receive a high score?
- **Confidence:** How reliable is the available evidence?
- **What could make it wrong:** Which missing context or data limitation could invalidate the recommendation?

The score prioritizes review opportunities. It does not establish that every selected item will benefit from a metadata change.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
top20 = queue[queue["qualifies"]].head(20).copy()

top20["confidence"] = np.select(
    [
        top20["active_days"] >= 20,
        top20["active_days"] >= 10
    ],
    [
        "Higher: at least 20 active days",
        "Medium: 10-19 active days"
    ],
    default="Lower: fewer than 10 active days"
)

top20["what_could_make_it_wrong"] = np.where(
    top20["active_days"] < 10,
    "Few active days may make the monthly metrics unstable.",
    "Query mix, search intent, or competition may explain the CTR gap."
)

review_columns = [
    "rank",
    "content_hash_id",
    "impressions",
    "avg_position",
    "march_ctr",
    "expected_ctr",
    "score",
    "action",
    "reason_code",
    "confidence",
    "what_could_make_it_wrong"
]

display(top20[review_columns].round(3))

,rank,content_hash_id,impressions,avg_position,march_ctr,expected_ctr,score,action,reason_code,confidence,what_could_make_it_wrong
88684,1,content_44f34c0a90047651,212404.0,0.666,0.011,0.069,12262.034,review_title_meta,high_visibility_low_ctr,Higher: at least 20 active days,"Query mix, search intent, or competition may e..."
17165,2,content_8e1334d6356668e3,134984.0,2.693,0.001,0.069,9217.809,review_title_meta,high_visibility_low_ctr,Higher: at least 20 active days,"Query mix, search intent, or competition may e..."
162352,3,content_fec55986a1868d62,124075.0,0.308,0.001,0.069,8464.772,review_title_meta,high_visibility_low_ctr,Higher: at least 20 active days,"Query mix, search intent, or competition may e..."
49729,4,content_9c057b66c30a3abb,83832.0,0.116,0.001,0.069,5686.838,review_title_meta,high_visibility_low_ctr,Higher: at least 20 active days,"Query mix, search intent, or competition may e..."
53756,5,content_bf078007df823490,44707.0,1.400,0.000,0.069,3086.079,review_title_meta,high_visibility_low_ctr,Higher: at least 20 active days,"Query mix, search intent, or competition may e..."
173139,6,content_d61fc394d10cba41,38000.0,2.363,0.003,0.069,2523.102,review_title_meta,high_visibility_low_ctr,Higher: at least 20 active days,"Query mix, search intent, or competition may e..."
99961,7,content_fc67675904376267,60172.0,2.126,0.030,0.069,2353.613,review_title_meta,high_visibility_low_ctr,Higher: at least 20 active days,"Query mix, search intent, or competition may e..."
149379,8,content_306bc78dff1eb683,80821.0,1.444,0.043,0.069,2078.992,review_title_meta,high_visibility_low_ctr,Higher: at least 20 active days,"Query mix, search intent, or competition may e..."
41069,9,content_dc91779c3d085398,25625.0,2.389,0.004,0.069,1668.868,review_title_meta,high_visibility_low_ctr,Higher: at least 20 active days,"Query mix, search intent, or competition may e..."
62516,10,content_757b1fa67827358d,31575.0,2.855,0.019,0.069,1579.590,review_title_meta,high_visibility_low_ctr,Higher: at least 20 active days,"Query mix, search intent, or competition may e..."


### Top-20 Manual Review

1. **Item:** [content_hash_id]
   - Action: Review title and meta description.
   - Why it is here: [Use its impressions and CTR gap.]
   - Confidence: [High / Medium / Low, with a reason.]
   - What would make it wrong: [A specific limitation.]

2. **Item:** [content_hash_id]
   - Action: Review title and meta description.
   - Why it is here: [Use its impressions and CTR gap.]
   - Confidence: [High / Medium / Low, with a reason.]
   - What would make it wrong: [A specific limitation.]

[Continue for all 20 items.]

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

## 4. Weak Picks and Leakage Check

A high score does not guarantee that an item is a good recommendation.

Potentially weak picks include items with:

- Very few active days.
- A small CTR gap despite high impressions.
- Unusual search-position values.
- A CTR difference that may be explained by query mix, search intent, or competition.

I will inspect the weakest selections and explain why they may not deserve immediate action.

### Leakage Check

The baseline uses only March 2026 impressions, clicks, average position, CTR, and active days.

It does not use April performance, the Week 3 outcome label, June data, or any product-generated flags.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
selected = queue[queue["qualifies"]].copy()

# Find five potentially weak selections:
# items with the fewest active days.
weak_picks = selected.sort_values(
    ["active_days", "score"],
    ascending=[True, True]
).head(5)

print("Five potentially weak picks:")
display(
    weak_picks[
        [
            "content_hash_id",
            "impressions",
            "active_days",
            "avg_position",
            "march_ctr",
            "expected_ctr",
            "score"
        ]
    ].round(3)
)

Five potentially weak picks:


,content_hash_id,impressions,active_days,avg_position,march_ctr,expected_ctr,score
137530,content_5fcf0ce19e5b3168,1060.0,4,1.188,0.000,0.069,73.171
50376,content_35d13a4f9286b796,1534.0,8,1.507,0.065,0.069,5.890
84287,content_bb3d66e6231a1687,1703.0,8,1.086,0.059,0.069,17.556
33682,content_b915a828eb6c562b,1226.0,8,0.432,0.000,0.069,84.630
6010,content_ec305a4a84cdb340,1379.0,9,0.071,0.000,0.069,95.191


In [9]:
# The only original measurement inputs used by the rule
allowed_features = [
    "impressions",
    "clicks",
    "avg_position",
    "march_ctr",
    "active_days"
]

# Check that future labels and forbidden fields are absent
forbidden_terms = [
    "position_worsened",
    "april",
    "june",
    "label",
    "product_flag"
]

leakage_columns = [
    col for col in queue.columns
    if any(term in col.lower() for term in forbidden_terms)
]

print("Allowed rule features:", allowed_features)
print("Forbidden/future columns found:", leakage_columns)

assert len(leakage_columns) == 0, "Potential leakage detected!"
assert "position_worsened" not in queue.columns

print("Leakage check passed!")

Allowed rule features: ['impressions', 'clicks', 'avg_position', 'march_ctr', 'active_days']
Forbidden/future columns found: []
Leakage check passed!


### Weak Pick Findings

1. **Item:** [content_hash_id]
   - Why it may be a weak pick: [Explain using its actual metrics.]

2. **Item:** [content_hash_id]
   - Why it may be a weak pick: [Explain using its actual metrics.]

3. **Item:** [content_hash_id]
   - Why it may be a weak pick: [Explain using its actual metrics.]

The baseline is useful for prioritizing investigation, but the ranked queue should not be treated as a definitive list of pages that need changes.

## Self-check

- [x] Selected Lane 1: Ranking Signal Analysis.
- [x] Checked two signals using bucket tables with sample counts.
- [ ] Assigned one-word verdicts with evidence for both signals.
- [x] Defined one transparent scoring rule.
- [x] Created one reason code and one action label.
- [x] Ranked the content items by score.
- [x] Generated `work/outputs/baseline_action_score.csv`.
- [ ] Reviewed the top 20 items individually.
- [ ] Identified and explained weak picks.
- [x] Checked for future-window and label-derived inputs.
- [x] Used March 2026 data only.
- [ ] Executed the entire notebook from top to bottom.
- [ ] Committed the notebook to GitHub.